### Assignment 3: Tracking a Panda: When Smoothers Fail (and How to Fix Them)

##### Group 6: Pavel Kuznetsov, Olga Tiupina, Sofya Konstantinova, Skoltech, 2025

In [ ]:
import numpy as np
import math
import matplotlib.pyplot as plt


def generate_true_trajectory(num, sigma, initial_condition):
    """
    Generate a panda walking trajectory using random walk model
    """
    X = np.zeros(num)
    X[0] = initial_condition
    
    # normally distributed random noise with a mean of zero and a variance of 𝜎𝑤^2
    w = np.random.normal(0, np.sqrt(sigma), num)
    
    # the random walk model
    for i in range(1, num):
        X[i] = X[i-1] + w[i-1]
    
    return X, w

def generate_measurements(true_trajectory, sigma):
    """
    Generate a panda walking measurements using random walk model
    """

    num = len(true_trajectory)

    z = np.zeros(num)
    
    n = np.random.normal(0, np.sqrt(sigma), num)    
    
    for i in range(num):
        z[i] = true_trajectory[i] + n[i]
    
    return z, n

def generate_measurements_withot_n(true_trajectory, sigma):
    """
    Generate a panda walking measurements using random walk model
    """

    num = len(true_trajectory)

    z = np.zeros(num)
    
    n = np.random.normal(0, np.sqrt(sigma), num)    
    
    for i in range(num):
        z[i] = true_trajectory[i] + n[i]
    
    return z


def M_determination(alpha):
    """
    Find window size M
    """
    M = (2.0 - alpha) / alpha
    return int(round(M))

def calculate_optimal_smoothing(sigma_w, sigma_n):
   
    chi = sigma_w / sigma_n
    alpha = (-chi + np.sqrt(chi ** 2 + 4 * chi)) / 2
    return alpha

def running_mean(M, measurements):
    """
    Running mean
    
    Parameters:
    M - window size
    z - input array
    
    Returns:
    x - array 
    """
    n = len(measurements)
    x = np.zeros(n)  
    half = (M - 1) // 2  
    
    #leftpart
    for i in range(half):
        x[i] = np.mean(measurements[:i + half + 1])
    
    #centralpart
    for i in range(half, n - half):
        window_sum = 0
        for k in range(i - half, i + half + 1):
            window_sum += measurements[k]
        x[i] = window_sum / M
    
    #rightpart
    for i in range(n - half, n):
        x[i] = np.mean(measurements[i - half:])
    
    return x


def generate_exponential_smoothing(measurements, alfa):
    """
    Generate Exponential smoothing
    """
    num = len(measurements)
    X_ex = np.zeros(num)
    X_ex[0] = measurements[0]
    
    # exponential smoothing
    for i in range(1, num):
        X_ex[i] = alfa * measurements[i] + (1 - alfa) * X_ex[i-1] 
    
    return X_ex

def generate_backward_exponential_smoothing(X_f, alfa):
    num = len(X_f)
    X_b = np.zeros(num)
    X_b[-1] = X_f[-1]

    # backward exponential smoothing
    for i in range(num - 2, -1, -1):
        X_b[i] = X_b[i + 1] + alfa * (X_f[i] - X_b[i + 1])
    
    return X_b

def deviation_indicator(x, z):
    sum = 0
    for i in range(0, len(x)):
        sum += ((x[i] - z[i])**2)
    return sum

def variability_indicator(x):
    sum = 0
    for i in range(0, len(x) - 2):
        sum += ((x[i + 2] - 2*x[i + 1] + x[i])**2)
    return sum

def generate_trajectory_with_accel(variance, points_amount, x0, v0, time_step):
    '''
    Trajectory generator: 
    x[i] = x[i-1] + v[i-1]*T + 0.5 * a[i-1]*T^2
    '''
    accel = np.random.normal(0, np.sqrt(variance), points_amount)
    
    v = np.zeros(points_amount)
    v[0] = v0
    x = np.zeros(points_amount)
    x[0] = x0

    for i in range(1, points_amount):
        v[i] = v[i - 1] + accel[i - 1] * time_step
        x[i] = x[i - 1] + v[i - 1] * time_step + 0.5 * accel[i - 1] * time_step**2
    return x

def generate_cyclic_trajectory(num, sigma, initial_condition, T):

    X = np.zeros(num)
    A = np.zeros(num)
    A[0] = initial_condition
    frequency = (2 * math.pi) / T

    # normally distributed random noise with a mean of zero and a variance of 𝜎𝑤^2
    w = np.random.normal(0, np.sqrt(sigma), num)
    
    # A
    for i in range(1, num):
        A[i] = A[i-1] + w[i]

    for i in range(1, num):
        X[i] = A[i] * np.sin(frequency * i + 3)

    return X

In [ ]:
fontsize_title = 30
fontsize_label = 20
fontsize_leg = 18

In our previous assignment, we observed a shift in estimates with forward exponential smoothing. To reduce the measurement error, it was decided to apply backward exponential smoothing to the forward exponential smoothing line

In [ ]:
# Parameters
initial_condition = 10
sigma_w = 28 ** 2
sigma_n = 97 ** 2

# generate trajectory
calm_true_trajectory, w = generate_true_trajectory(num=300, sigma=sigma_w, initial_condition=initial_condition)
calm_measurements, n = generate_measurements(true_trajectory=calm_true_trajectory, sigma=sigma_n)

alpha = calculate_optimal_smoothing(sigma_w, sigma_n)
print(f"Optimal smoothing coefficient: {float(alpha):.4f}")

calm_forward_exp_smoothing = generate_exponential_smoothing(calm_measurements, alpha)
calm_backward_exp_smoothing = generate_backward_exponential_smoothing(calm_forward_exp_smoothing, alpha)

M = M_determination(alpha=alpha)
x_mean = running_mean(M, calm_measurements)

In [ ]:
plt.figure(figsize=(25, 18))

plt.subplot(2, 1, 1)
plt.plot(calm_true_trajectory, linestyle='--', color="black", linewidth=2, alpha=1, label='True trajectory')
plt.plot(calm_measurements, color='blue', linewidth=0.25, alpha=1, label='Measurements')
plt.plot(calm_forward_exp_smoothing, color='green', linewidth=2, alpha=1, label='Forward Exponential Smoothing')
plt.plot(calm_backward_exp_smoothing, color='red', linewidth=3, alpha=1, label='Backward Exponential Smoothing')
plt.plot(x_mean, color='orange', linewidth=1.5, alpha=1, label='Running Mean')
plt.title('\n Fig.1. Calm Walk: Measurements, True Trajectory, Forward and Backward Exponential Smoothing and Running mean \n', fontsize=fontsize_title)
plt.xlabel('\n Time step \n', fontsize=fontsize_label)
plt.ylabel('\n Position, m \n', fontsize=fontsize_label)
plt.xlim(-5, 305)
plt.grid(True, alpha=0.3)
plt.legend(fontsize=fontsize_leg)

plt.subplot(2, 1, 2)
plt.plot(calm_true_trajectory, linestyle='--', color="black", linewidth=2, alpha=1, label='True trajectory')
plt.plot(calm_measurements, color='blue', linewidth=0.25, alpha=1, label='Measurements')
plt.plot(calm_forward_exp_smoothing, color='green', linewidth=2, alpha=1, label='Forward Exponential Smoothing')
plt.plot(calm_backward_exp_smoothing, color='red', linewidth=3, alpha=1, label='Backward Exponential Smoothing')
plt.plot(x_mean, color='orange', linewidth=1.5, alpha=1, label='Running Mean')
plt.title('\n Fig.2. Calm Walk: Measurements, True Trajectory, Forward and Backward Exponential Smoothing and Running mean with zooming \n', fontsize=fontsize_title)
plt.xlabel('\n Time step \n', fontsize=fontsize_label)
plt.ylabel('\n Position, m \n', fontsize=fontsize_label)
plt.xlim(50, 150)
plt.grid(True, alpha=0.3)
plt.legend(fontsize=fontsize_leg)
plt.tight_layout()
plt.show()

According on a visual comparison of the graphs obtained, we can draw the following conclusion, that applying backward exponential smoothing after using forward exponential smoothing:
- Eliminates the delay and improves accuracy (compared to forward exponential smoothing)
- Provides a smoother estimate (compared to the Running mean)

We use deviation and variability indicators for a more rigorous comparison of assessment results:

In [ ]:
dev_coef_running = deviation_indicator(calm_measurements, x_mean)
dev_coef_backward_exp = deviation_indicator(calm_measurements, calm_backward_exp_smoothing)

print(f"Deviation indicators for Running Mean: {dev_coef_running}")
print(f"Deviation indicators for Backward Exponential Smoothing: {dev_coef_backward_exp}")

var_coef_running = variability_indicator(x_mean)
var_coef_backward_exp = variability_indicator(calm_backward_exp_smoothing)

print(f"Variability indicators for Running Mean: {var_coef_running}")
print(f"Variability indicators for Backward Exponential Smoothing: {var_coef_backward_exp}")

The deviation indicators for the Running mean and the Backward exponential smoothing turned out to be approximately comparable. This indicates a comparable level of smoothing when using these methods

The variability indicators with Backward exponential smoothing turned out to be an order of magnitude lower than with the Running mean. This indicates more efficient smoothing when using Backward exponential smoothing

We were test the Running mean in two realistic grove scenarios and contrast with exponential smoothing:

In [ ]:
# initialization
acc_variance = 10
points_amount = 300
x0 = 5
v0 = 0
time_step = 0.1

noise_variance = 500

# built trajectory
x_acc = generate_trajectory_with_accel(acc_variance, points_amount, x0, v0, time_step)
z_acc = generate_measurements_withot_n(x_acc, noise_variance)

In [ ]:
M_values = list(range(35, 103, 4))
num_plots = (len(M_values) + 2) // 3

for plot_idx in range(num_plots):
    plt.figure(figsize=(25, 8))
    
    plt.plot(x_acc, color="black", linewidth=2, alpha=0.8, label='True trajectory')
    plt.plot(z_acc, linewidth=0.5, alpha=1, label='Measurements')
    
    start_idx = plot_idx * 3
    end_idx = min(start_idx + 3, len(M_values))
    current_M_values = M_values[start_idx:end_idx]
    
    for M in current_M_values:
        smoothing_acc_trajectory_RM = running_mean(M, z_acc)
        plt.plot(smoothing_acc_trajectory_RM, linewidth=1, alpha=1, label=f'RM M={M}')
    
    plt.title(f'\n Fig.{plot_idx + 3}. Sun-nap: Measurements, True Trajectory - 300 points (M = {current_M_values}) \n', fontsize=fontsize_title)
    plt.xlabel('\n Time step \n', fontsize=fontsize_label)
    plt.ylabel('\n Position, m \n', fontsize=fontsize_label)
    plt.grid(True, alpha=0.3)
    plt.legend(fontsize=fontsize_leg)
    plt.tight_layout()
    plt.show()

Parameter Selection for Moving Average Smoothing

The running mean  (RM)  smoothing technique was evaluated across a range of window sizes to determine the optimal value. The selection process was guided by both the nature of the data and visual analysis of the resulting smoothed trajectories.

The underlying true trajectory is predominantly linear, but the measurements are contaminated with significant noise. This context is critical for parameter selection: a small window size inadequately suppresses noise, leaving a high estimation error, while an excessively large window may distort the genuine signal. To navigate this trade-off, we focused on larger window sizes to effectively attenuate the noise. A sweep of window sizes from 35 to 99 was performed and the results were visualized.

Based on a visual assessment of the smoothed trajectories, a window size of RM = 95 was identified as optimal. This setting successfully smooths the pronounced measurement noise while faithfully preserving the fundamental linear trend of the true trajectory.

Next, we visually evaluate the smoothing factor for exponential smoothing.

In [ ]:
alphas = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6]

plt.figure(figsize=(25, 18))

plt.subplot(2, 1, 1)
plt.plot(x_acc, color="black", linewidth=2, alpha=1, label='True trajectory')
plt.plot(z_acc, color="blue", linewidth=0.5, alpha=1, label='Measurements')

for a in alphas:
    smoothing_acc_trajectory_ES = generate_exponential_smoothing(z_acc, a)
    plt.plot(smoothing_acc_trajectory_ES, linewidth=0.8, alpha=0.8, label=f'ES α={a}')

plt.title('\n Fig.9. Sun-nap: Measurements, True Trajectory - 300 points (different α) \n', fontsize=fontsize_title)
plt.xlabel('\n Time step \n', fontsize=fontsize_label)
plt.ylabel('\n Position, m \n', fontsize=fontsize_label)
plt.grid(True, alpha=0.3)
plt.legend(fontsize=fontsize_leg)

plt.subplot(2, 1, 2)
plt.plot(x_acc, color="black", linewidth=2, alpha=1, label='True trajectory')
plt.plot(z_acc, color="blue", linewidth=0.5, alpha=1, label='Measurements')

for a in alphas:
    smoothing_acc_trajectory_ES = generate_exponential_smoothing(z_acc, a)
    plt.plot(smoothing_acc_trajectory_ES, linewidth=1, alpha=0.8, label=f'ES α={a}')

plt.title('\n Fig.10. Sun-nap: Measurements, True Trajectory - 300 points (different α) with zooming \n', fontsize=fontsize_title)
plt.xlabel('\n Time step \n', fontsize=fontsize_label)
plt.ylabel('\n Position, m \n', fontsize=fontsize_label)
plt.xlim(100, 150)
plt.ylim(-30, 80)
plt.grid(True, alpha=0.3)
plt.legend(fontsize=fontsize_leg)

plt.tight_layout()
plt.show()

The analysis reveals that the smoothing performance deteriorates significantly with larger values of the smoothing parameter α. This behavior can be explained by the fundamental mechanism of exponential smoothing, which combines two components: current measurements and historical estimates.

The exponential smoothing algorithm computes estimates as a weighted combination:

Current measurements multiplied by the smoothing coefficient α

Historical estimates multiplied by the complementary coefficient (1 - α)

Given the high noise levels present in the measurement data, excessive reliance on current measurements proves detrimental to estimation quality. The optimal strategy therefore emphasizes greater dependence on the historical trajectory, which corresponds to selecting smaller α values.

Through systematic evaluation, α = 0.3 emerges as the optimal parameter choice, striking the best balance between responsiveness to true signal changes and robustness against measurement noise. However, the visual analysis of the trajectory plot reveals that the performance differences between neighboring α values are somewhat subtle, making the optimal selection less distinctly apparent. 

But for this graph, the optimal alpha is not so clear, so the characteristics were built for it

In [ ]:
smoothing_acc_trajectory_ES = []
deviation_indicators = []
variability_indicators = []
alphas = np.arange(0.1, 1.0, 0.01)

for i in range(0, len(alphas)):
    try:
        smoothed_data =  generate_exponential_smoothing(z_acc, alphas[i])
        smoothing_acc_trajectory_ES.append(smoothed_data)
        original_cut = z_acc[:len(smoothed_data)]
        
        dev_coef = deviation_indicator(smoothed_data, original_cut)
        var_coef = variability_indicator(smoothed_data)
        
        deviation_indicators.append(dev_coef)
        variability_indicators.append(var_coef)
        
    except Exception as e:
        print(f"Error for alpha={i}: {e}")
        continue

deviation_indicators = np.array(deviation_indicators)
variability_indicators = np.array(variability_indicators)

In [ ]:
plt.figure(figsize=(25, 10))

plt.plot(alphas, deviation_indicators, color="blue", linewidth=2, alpha=0.8, label='Deviation indicator')
plt.plot(alphas, variability_indicators, color="red", linewidth=2, alpha=0.8, label='Variability indicator')
plt.title('\n Fig.11. Optimal Smoothing Coefficient Selection \n', fontsize=fontsize_title)
plt.xlabel('\n Smoothing Coefficient (alpha) \n', fontsize=fontsize_label)
plt.xticks(np.arange(0, 1.0, 0.05))
plt.ylabel('\n Indicator Value \n', fontsize=fontsize_label)
plt.grid(True, alpha=0.3)
plt.legend(fontsize=fontsize_leg)
plt.show()

By the plot optimal coefficient is alpha = 0.38

In [ ]:
# find deviation and variability indicators for choosen alpha and M
M_choosen = 95
alpha_choosen = 0.38


smoothed_data_RM = running_mean(M_choosen, z_acc)
smoothed_data_ES =  generate_exponential_smoothing(z_acc, alpha_choosen)

dev_coef_RM = deviation_indicator(smoothed_data_RM, z_acc)
dev_coef_ES = deviation_indicator(smoothed_data_ES, z_acc)

print(f'deviation indicator for ES is {dev_coef_ES}')
print(f'deviation indicator for RM is {dev_coef_RM}')

var_coef_RM = variability_indicator(smoothed_data_RM)
var_coef_ES = variability_indicator(smoothed_data_ES)

print(f'variability indicator for RM is {var_coef_RM}')
print(f'variability indicator for ES is {var_coef_ES}')

According to the received indicator values:

By variability indicator, the running mean yields a significantly better result. By deviation indicator the exponential smoothing is better, but not significantly so. Therefore, the running mean performs better.

In [ ]:
plt.figure(figsize=(25, 10))

# Plot for 300 points
plt.plot(x_acc, color="black", linewidth=2, alpha=1, label='True trajectory')
plt.plot(z_acc, color="blue", linewidth=0.5, alpha=1, label='Measurements')
plt.plot(smoothed_data_RM, color="red", linewidth=1.5, alpha=1, label='Running Mean')
plt.plot(smoothed_data_ES, color="green", linewidth=1.5, alpha=1, label='Exponential Smoothing')
plt.title('\n Fig.12. Sun-nap: Measurements, True Trajectory, Running Mean and Exponential Smoothing - 300 points \n', fontsize=fontsize_title)
plt.xlabel('\n Time step \n', fontsize=fontsize_label)
plt.ylabel('\n Position, m \n', fontsize=fontsize_label)
plt.grid(True, alpha=0.3)
plt.legend(fontsize=fontsize_leg)

plt.tight_layout()
plt.show()

It is also noticeable from the visualization that the running mean gives a better result than exponential smoothing.

Generate a cyclic trajectory 𝑋𝑖 based on the following equation
𝑋𝑖 = 𝐴𝑖 ⋅ sin (𝜔𝑖 + 3)

In [ ]:
cyclic_trajectory = generate_cyclic_trajectory(num= 200, sigma = 0.08 ** 2, initial_condition = 1, T=32) 

In [ ]:
plt.figure(figsize=(25,8))

plt.plot(cyclic_trajectory, linewidth=1, color='red')
plt.title('\n Fig.13. Cyclic trajectory \n', fontsize=fontsize_title)
plt.xlabel('\n Steps \n', fontsize=fontsize_label)
plt.ylabel('\n Position, m \n', fontsize=fontsize_label)
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

Generate measurements 𝑧𝑖 of the panda’s cyclic motion 𝑋𝑖 
𝑧𝑖 = 𝑋𝑖 + 𝜂𝑖,
𝜂𝑖 is normally distributed random noise with a mean of zero and a variance of 𝜎𝜂^2 = 0.05

In [ ]:
measurements = generate_measurements_withot_n(cyclic_trajectory, sigma = 0.05)

In [ ]:
plt.figure(figsize=(25, 8))

plt.plot(measurements,linewidth=1, color='blue', alpha=0.8, label = 'Measurements')
plt.plot(cyclic_trajectory, linewidth=2, color='red', alpha=1, label = "Cyclic trajectory")
plt.title('\n Fig.14. Measurements and cyclic trajectory \n', fontsize=fontsize_title)
plt.xlabel('\n Steps \n', fontsize=fontsize_label)
plt.ylabel('\n Position, m \n', fontsize=fontsize_label)
plt.grid(True, alpha=0.3)
plt.legend(fontsize=fontsize_leg)

plt.tight_layout()
plt.show()

Running mean with a window size of 𝑀 = 13 to the measurements 𝑧𝑖 

In [ ]:
x_mean_13 = running_mean(M = 13, measurements = measurements)

In [ ]:
plt.figure(figsize=(25, 8))

plt.plot(x_mean_13, linewidth=1, color='black', alpha=1, label ="Runnung mean, M = 13")
plt.plot(measurements, linewidth=1, color='blue', alpha=1, label = 'Measurements')
plt.plot(cyclic_trajectory, linewidth=2, color='red', alpha=1, label = "Cyclic trajectory")
plt.title('\n Fig.15. Measurements, cyclic trajectory and runnung mean (M = 13 and T = 32) \n', fontsize=fontsize_title)
plt.xlabel('\n Steps \n', fontsize=fontsize_label)
plt.ylabel('\n Position, m \n', fontsize=fontsize_label)
plt.grid(True, alpha=0.3)
plt.legend(fontsize=fontsize_leg)

plt.tight_layout()
plt.show()

According on the graph obtained, it can be seen that the estimate for a given window size does not change its character and has a drawdown of values in the peaks area

Running mean with a window size of 𝑀 = 25 to the measurements 𝑧𝑖 and T = 32 st

In [ ]:
x_mean_25 = running_mean(M = 25, measurements = measurements)

In [ ]:
plt.figure(figsize=(25, 8))

plt.plot(x_mean_25, linewidth=1, color='black', alpha=1, label ="Runnung mean, M = 25")
plt.plot(measurements, linewidth=1, color='blue', alpha=1, label = 'Measurements, T = 32 st')
plt.plot(cyclic_trajectory, linewidth=2, color='red', alpha=1, label = "Cyclic trajectory, T = 32 st")
plt.title('\n Fig.16. Measurements, cyclic trajectory and running mean (M = 25 and T = 32) \n', fontsize=fontsize_title)
plt.xlabel('\n Steps \n', fontsize=fontsize_label)
plt.ylabel('\n Position, m \n', fontsize=fontsize_label)
plt.grid(True, alpha=0.3)
plt.legend(fontsize=fontsize_leg)

plt.tight_layout()
plt.show()

With these ratios of the period and the size of the window, changes the oscillations insignificantly

Running mean with a window size of 𝑀 = 25 to the measurements 𝑧𝑖 and T = 16 st (1,5T = M).
Inverse oscillations

In [ ]:
# inverse oscillations
cyclic_trajectory_25 = generate_cyclic_trajectory(num= 200, sigma = 0.08 ** 2, initial_condition = 1, T=16) 
measurements_25 = generate_measurements_withot_n(cyclic_trajectory_25, sigma = 0.05)
x_mean_25 = running_mean(M = 25, measurements = measurements_25)

In [ ]:
plt.figure(figsize=(25, 8))

plt.plot(x_mean_25, linewidth=1, color='black', alpha=1, label ="Runnung mean, M = 25")
plt.plot(measurements_25, linewidth=1, color='blue', alpha=1, label = 'Measurements, T = 16 st')
plt.plot(cyclic_trajectory_25, linewidth=2, color='red', alpha=1, label = "Cyclic trajectory, T = 16 st")
plt.title('\n Fig.17. Measurements, cyclic trajectory and running mean (M = 25 and T = 16) \n', fontsize=fontsize_title)
plt.xlabel('\n Steps \n', fontsize=fontsize_label)
plt.ylabel('\n Position, m \n', fontsize=fontsize_label)
plt.grid(True, alpha=0.3)
plt.legend(fontsize=fontsize_leg)

plt.tight_layout()
plt.show()

With these ratios of the period and the size of the window, produces inverse oscillations

Running mean with a window size of 𝑀 = 25 to the measurements 𝑧𝑖 and T = 25 st (T = M).
The loss of oscillations

In [ ]:
# the loss of oscillations
cyclic_trajectory_25 = generate_cyclic_trajectory(num= 200, sigma = 0.08 ** 2, initial_condition = 1, T=25) 
measurements_25 = generate_measurements_withot_n(cyclic_trajectory_25, sigma = 0.05)
x_mean_25 = running_mean(M = 25, measurements = measurements_25)

In [ ]:
plt.figure(figsize=(25, 8))

plt.plot(x_mean_25, linewidth=1, color='black', alpha=1, label ="Runnung mean, M = 25")
plt.plot(measurements_25, linewidth=1, color='blue', alpha=1, label = 'Measurements, T = 25 st')
plt.plot(cyclic_trajectory_25, linewidth=2, color='red', alpha=1, label = "Cyclic trajectory, T = 25 st")
plt.title('\n Fig.18. Measurements, cyclic trajectory and running mean (M = 25 and T = 25) \n', fontsize=fontsize_title)
plt.xlabel('\n Steps \n', fontsize=fontsize_label)
plt.ylabel('\n Position, m \n', fontsize=fontsize_label)
plt.grid(True, alpha=0.3)
plt.legend(fontsize=fontsize_leg)

plt.tight_layout()
plt.show()

With these ratios of the period and the size of the window, leads to the loss of oscillations (resulting in zero oscillations)

Running mean with a window size of 𝑀 = 25 to the measurements 𝑧𝑖 and T = 25 st (T >> M).
Changes the oscillations insignificantly

In [ ]:
# Changes the oscillations insignificantly
cyclic_trajectory_25 = generate_cyclic_trajectory(num= 200, sigma = 0.08 ** 2, initial_condition = 1, T=70) 
measurements_25 = generate_measurements_withot_n(cyclic_trajectory_25, sigma = 0.05)
x_mean_25 = running_mean(M = 25, measurements = measurements_25)

In [ ]:
plt.figure(figsize=(25, 8))

plt.plot(x_mean_25, linewidth=1, color='black', alpha=1, label ="Runnung mean, M = 25")
plt.plot(measurements_25, linewidth=1, color='blue', alpha=1, label = 'Measurements, T = 70 st')
plt.plot(cyclic_trajectory_25, linewidth=2, color='red', alpha=1, label = "Cyclic trajectory, T = 70 st")
plt.title('\n Fig.19. Measurements, cyclic trajectory and running mean (M = 25 and T = 70) \n', fontsize=fontsize_title)
plt.xlabel('\n Steps \n', fontsize=fontsize_label)
plt.ylabel('\n Position, m \n', fontsize=fontsize_label)
plt.grid(True, alpha=0.3)
plt.legend(fontsize=fontsize_leg)

plt.tight_layout()
plt.show()

With these ratios of the period and the size of the window, changes the oscillations insignificantly

#### Conclusion

During this assignment, we:
-  observed a shift (delay) in forward exponential smoothing estimates for random-walk process. Applied backward exponential smoothing to forward exponential smoothing curve to further reduce measurement errors, compared methods visually by plotting the true trajectory of the panda, measurements, running mean, and backward exponential mean. Backward exponential smoothing successfully eliminated the lag and provided a smoother estimate than the running mean.
-  compared the estimation results of the running mean and backward exponential smoothing using deviation and variability indicators. The deviation indicators for both methods were comparable.The variability indicator for backward smoothing was an order of magnitude lower, proving it to be a more efficient and stable smoothing method.
- generated true trajectory with gradual velocity changes, along with very noisy measurements,empirically determined optimal parameters:running mean window (M=95) and exponential smoothing coefficient (α=0.38).The two methods were compared using deviation and variability indicators. The running mean provided a better result than exponential smoothing. Visualization also confirmed that the running mean was more effective.
- generated a cyclic trajectory, measurements of the panda’s cyclic motion,applied running mean with M=13, showing a slight drawdown at the peaks. 
- determined the period of oscillations for which applying a running mean with window size 𝑀 = 25:
  - M ≈ 1.5T - inverse oscillations
  - M = T - the loss of oscillations
  - M << T - changes the oscillations insignificantly.


The analysis demonstrates that no single smoothing method is universally best. For tracking problems with lag, forward-backward exponential smoothing is superior, while for denoising linear trends(like gradually changing velocity), the running mean excels. For oscillating signals, the running mean only works well if its window size is properly matched to the oscillation period. The wrong size will create an inaccurate picture of the cycles.
